## SETUP

In [ ]:
# only run if first time using
# for newer versions of DataFrames you need to edit the files directly (WE NEED TO PUSH NEW!)
using Pkg
Pkg.build("Blink")
Pkg.add(url="https://github.com/paden1991/CurricularAnalytics.jl") # delete original packs from computer !
Pkg.add(url="https://github.com/paden1991/CurricularVisualization.jl")

In [1]:
# ---------------------------------------------------------
# 1. SETUP
# ---------------------------------------------------------
using CurricularAnalytics
using CSV
using DataFrames
using PrettyTables
using CurricularVisualization
# surgical patch to strip 'splitcols' only for DataFrames
import PrettyTables: _pretty_table

 
#function PrettyTables._pretty_table(io::IO, data::DataFrames.AbstractDataFrame, backend::Symbol; splitcols=false, kwargs...)

function PrettyTables._pretty_table(io::IO, data::DataFrames.AbstractDataFrame, backend::Symbol; splitcols=false, kwargs...)
    # This method is more specific than the library's (which uses data::Any).
    # 'invoke' now correctly finds the original generic method instead of recursing.
    return invoke(_pretty_table, Tuple{IO, Any, Symbol}, io, data, backend; kwargs...)
end

 

println("PrettyTables surgical patch applied (Recursion fix).")
# This forces DataFrames to use a simpler printing method that avoids the 'splitcols' bug
ENV["DATAFRAMES_ROWS"] = 50
ENV["DATAFRAMES_COLUMNS"] = 20

WebIO._IJuliaInit()

PrettyTables surgical patch applied (Recursion fix).


20

# FIU

### TEST - IGNORE

In [ ]:
# Set your paths
desktop_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/FIU/"
degree_plan_file = desktop_path * "degree_plan_REVISE.csv"
grades_file      = desktop_path * "historical_grades.csv" # can use transfer equivalency database to map classes to other institutions as needed
vis_file = desktop_path * "degree_plan_vis.csv"
# Load degree plan (This automatically creates plan.curriculum!)
plan = read_csv(degree_plan_file) 
vis_plan = read_csv(vis_file) # anonymized grade output for each of the courses <- just grab courses in degree plan file 
grades_df = CSV.read(grades_file, DataFrame) # grades file we can try a) sub for different specific courses in deg plan, b) sub general course for courses meeting same requirements

In [ ]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

 

course_passrate = 0.85 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

 

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

 

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)

 

    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, grades_df)

 

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

 

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

 

println("Pass rates successfully injected into the degree plan!")

In [ ]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")
 
# two ways for simulation
num_students = 1000 # we can maybe use cohort from fall24?
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis

# enroll sequentially through the courses in the degree plan unless they 
max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

simulation = simulate(plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

In [ ]:
# ---------------------------------------------------------
# 4. GENERATE REPORT
# ---------------------------------------------------------
real_passrate = true   

# This function prints directly to your console!
simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)

 

#visualize(plan)

### GRADES FILE CONSOLIDATED

In [53]:
desktop_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/FIU/"
degree_plan_file = desktop_path * "degree_plan_REVISE.csv"
grades_file      = desktop_path * "slim_grades.csv" 
slim_grade = CSV.read(grades_file, DataFrame) # grades file we can try a) sub for different specific courses in deg plan, b) sub general course for courses meeting same requirements
plan = read_csv(degree_plan_file) 

DegreePlan("Generic", Curriculum(617800848, "Computer Science - BS", "Florida International University", "BS", semester, "11.0101", AbstractCourse[Course(5, Dict(617800848 => 1), "Social Science Group One [RQ4005/LN10]", 3, String3("SOC"), String7("UCC1"), String3("FIU"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(6, Dict(617800848 => 2), "Humanities - Group One [RQ4004/LN10]", 3, String3("HUM"), String7("UCC1"), String3("FIU"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(10, Dict(617800848 => 3), "UCC Elective", 3, "", "", String3("FIU"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(11, Dict(617800848 => 4), "Social Science Group Two [RQ4005/LN30]", 3, String3("SOC"), String7("UCC2"), String3("FIU"), "", "", Course[], "", Dict{Int64, Requisite}(), Learnin

In [54]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES 
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

course_passrate = 0.9 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)
    
    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, slim_grade)

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

println("Pass rates successfully injected into the degree plan!")

--- INJECTING HISTORICAL PASS RATES ---
Pass rates successfully injected into the degree plan!


In [55]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")

# two ways for simulation
num_students = 1000
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis
# enroll sequentially through the courses in the degree plan unless they 

max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

 

simulation = simulate(plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

--- RUNNING SIMULATION ---


Simulation(DegreePlan("Generic", Curriculum(617800848, "Computer Science - BS", "Florida International University", "BS", semester, "11.0101", AbstractCourse[Course(5, Dict(617800848 => 1), "Social Science Group One [RQ4005/LN10]", 3, String3("SOC"), String7("UCC1"), String3("FIU"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}("model" => Dict{Any, Any}(:passrate => 0.8638976886072979), "termenrollment" => [1000.0, 127.0, 21.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "id" => 1, "failures" => 168, "num_students_passes" => 14726, "term_req" => 0, "grades" => [4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0  …  0.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0], "enrolled" => 1148, "num_students_taken" => 17046, "students" => Student[]…), 0.8638976886072979), Course(6, Dict(617800848 => 2), "Humanities - Group One [RQ4004/LN10]", 3, String3("HUM"), String7("UCC1"), String3("FIU"), "", "", Course[], "", Dict{Int64, Requisite

In [56]:
real_passrate = true   

# This function prints directly to your console!
simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)


------------ Simulation Report ------------
Computer Science - BS, BS -- Generic

-------- Simulation Statistics --------
Number of terms: 12
Max Credits per Term: 18
Max Course Attempts: 3
Number of Students: 1000
Preset Course Pass Rates: 90.0%

-------- Graduation Statistics --------
Number of Students Graduated: 608
Graduation Rate: 60.8%
Term Graduation Rates: 
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.033, 0.347, 0.57, 0.605, 0.608, 0.0]
Average time to degree: 8.442434210526315 terms

-------- Stop out Statistics --------
Number of Students Stopped Out (Stopout Model Prediction + Reached Max Attempts): 392
Number of Students Reaching Max Attempts: 84
Stop-out Rate: 39.2%
Cumulative Term Stop-out Rates (including reached max course attempts students): 
[0.075, 0.204, 0.249, 0.325, 0.366, 0.381, 0.385, 0.391, 0.392, 0.392, 0.392, 0.0]

Cumulative Term Stop-out Rates (excluding reaching max course attempts students): 
[0.075, 0.204, 0.227, 0.276, 0.302, 0.306, 0.308, 0.308, 0.308, 0.308, 0

### TEST SIMULATION - START HERE FOR SIMULATION FILE

In [6]:
sim_df = DataFrame("SimID" => [], "Institution" => [], "Grad_Rate" => [], "Std_Max_Attempt" => [], "Time_To_Degree" => [])


# add row in df: push!(df,[6 "F"])
for i in 1:100
    sim = simulate(plan, max_course_attempts, cohort, max_credits = max_credits,  performance_model = PassRate, enrollment_model = Enrollment,
        duration = num_terms, duration_lock = duration_lock, stopouts = stopouts)
    push!(sim_df, [i, "FIU_OLD", sim.grad_rate, length(sim.reach_attempts_students), sim.time_to_degree])
end
# print(sim_df)

## NEW FIU CS CURRICULUM

In [7]:
degree_plan_file = desktop_path * "NEWCS_curric.csv"
grades_file      = desktop_path * "slim_grades2.csv" 
slim_grade = CSV.read(grades_file, DataFrame) # grades file we can try a) sub for different specific courses in deg plan, b) sub general course for courses meeting same requirements
plan = read_csv(degree_plan_file) 

DegreePlan("Generic", Curriculum(617800848, "Computer Science - BS", "Florida International University", "BS", semester, "11.0101", AbstractCourse[Course(5, Dict(617800848 => 1), "Social Science Group One [RQ4005/LN10]", 3, String3("SOC"), String7("UCC1"), String3("FIU"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(6, Dict(617800848 => 2), "Humanities - Group One [RQ4004/LN10]", 3, String3("HUM"), String7("UCC1"), String3("FIU"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(10, Dict(617800848 => 3), "UCC Elective", 3, "", "", String3("FIU"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(11, Dict(617800848 => 4), "Social Science Group Two [RQ4005/LN30]", 3, String3("SOC"), String7("UCC2"), String3("FIU"), "", "", Course[], "", Dict{Int64, Requisite}(), Learnin

In [8]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES 
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

course_passrate = 0.9 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)
    
    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, slim_grade)

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

println("Pass rates successfully injected into the degree plan!")

--- INJECTING HISTORICAL PASS RATES ---
Pass rates successfully injected into the degree plan!


In [9]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")

# two ways for simulation
num_students = 1000
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis
# enroll sequentially through the courses in the degree plan unless they 

max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

 

simulation = simulate(plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

--- RUNNING SIMULATION ---


Simulation(DegreePlan("Generic", Curriculum(617800848, "Computer Science - BS", "Florida International University", "BS", semester, "11.0101", AbstractCourse[Course(5, Dict(617800848 => 1), "Social Science Group One [RQ4005/LN10]", 3, String3("SOC"), String7("UCC1"), String3("FIU"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}("model" => Dict{Any, Any}(:passrate => 0.8638976886072979), "termenrollment" => [1000.0, 120.0, 14.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "id" => 1, "failures" => 147, "num_students_passes" => 14726, "term_req" => 0, "grades" => [4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 0.0, 4.0, 4.0, 0.0  …  4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0], "enrolled" => 1134, "num_students_taken" => 17046, "students" => Student[]…), 0.8638976886072979), Course(6, Dict(617800848 => 2), "Humanities - Group One [RQ4004/LN10]", 3, String3("HUM"), String7("UCC1"), String3("FIU"), "", "", Course[], "", Dict{Int64, Requisite

In [ ]:
real_passrate = true   

# This function prints directly to your consbbole!
simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)

### TEST SIMULATION

In [10]:
# TO COMPARE NEW AND OLD FIU
for i in 1:100
    sim = simulate(plan, max_course_attempts, cohort, max_credits = max_credits,  performance_model = PassRate, enrollment_model = Enrollment,
        duration = num_terms, duration_lock = duration_lock, stopouts = stopouts)
    push!(sim_df, [i, "FIU_NEW", sim.grad_rate, length(sim.reach_attempts_students), sim.time_to_degree])
end
# print(sim_df)

In [13]:
# to compare fiu curriculum with each other, everything will be together at the end 
simulation_results_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/" # <- get sim at very end!
old_vs_new_fiu = simulation_results_path * "compare_fiu_curric.csv"
CSV.write(old_vs_new_fiu, sim_df) 

"C:/Users/cpanc/OneDrive/Documents/IDC6940/compare_fiu_curric.csv"

# FAMU

In [11]:
desktop_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/FAMU/"
degree_plan_file = desktop_path * "degree_plan_famu.csv"
grades_file      = desktop_path * "slim_grades_famu.csv"
vis_famu = desktop_path * "degree_plan_famu_vis.csv"
# -----------------------------------------------------------
slim_grade = CSV.read(grades_file, DataFrame) 
famu_plan = read_csv(degree_plan_file)
#vis_famu = read_csv(vis_famu)
#visualize(vis_famu)

DegreePlan("Generic", Curriculum(87418211, "BS Computer Science", "Florida Agricultural and Mechanical University", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(87418211 => 1), "Biological Science", 3, String3("BSC"), String7("UCC1"), String7("FAMU"), "", "", Course[], String15("BSC1005"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(2, Dict(87418211 => 2), "Freshman Communication Skills I", 3, String3("ENC"), String7("1101"), String7("FAMU"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(3, Dict(87418211 => 3), "Calculus I", 4, String3("MAC"), String7("2311"), String7("FAMU"), "", "", Course[], String15("MAC2311"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(4, Dict(87418211 => 4), "Public Speaking", 3, String3("SPC"), String7("2608"), String7("FAMU"), "", "", Course[], St

In [12]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES 
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

course_passrate = 0.9 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in famu_plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)
    
    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, slim_grade)

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

println("Pass rates successfully injected into the degree plan!")

--- INJECTING HISTORICAL PASS RATES ---
Pass rates successfully injected into the degree plan!


In [14]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")

# two ways for simulation
num_students = 1000
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis
# enroll sequentially through the courses in the degree plan unless they 

max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

 

simulation = simulate(famu_plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

--- RUNNING SIMULATION ---


Simulation(DegreePlan("Generic", Curriculum(87418211, "BS Computer Science", "Florida Agricultural and Mechanical University", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(87418211 => 1), "Biological Science", 3, String3("BSC"), String7("UCC1"), String7("FAMU"), "", "", Course[], String15("BSC1005"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}("model" => Dict{Any, Any}(:passrate => 0.9), "termenrollment" => [1000.0, 97.0, 12.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "id" => 1, "failures" => 123, "num_students_passes" => "N/A", "term_req" => 0, "grades" => [4.0, 0.0, 4.0, 4.0, 4.0, 4.0, 0.0, 4.0, 4.0, 4.0  …  4.0, 4.0, 0.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0], "enrolled" => 1109, "num_students_taken" => "N/A", "students" => Student[]…), 0.9), Course(2, Dict(87418211 => 2), "Freshman Communication Skills I", 3, String3("ENC"), String7("1101"), String7("FAMU"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), Learni

In [ ]:
real_passrate = true   

simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)

## TEST - SIMULATIONS

In [15]:
for i in 1:100
    sim = simulate(famu_plan, max_course_attempts, cohort, max_credits = max_credits,  performance_model = PassRate, enrollment_model = Enrollment,
        duration = num_terms, duration_lock = duration_lock, stopouts = stopouts)
    push!(sim_df, [i, "FAMU", sim.grad_rate, length(sim.reach_attempts_students), sim.time_to_degree])
end
print(sim_df)

300×5 DataFrame
 Row │ SimID  Institution  Grad_Rate  Std_Max_Attempt  Time_To_Degree 
     │ Any    Any          Any        Any              Any            
─────┼────────────────────────────────────────────────────────────────
   1 │ 1      FIU_OLD      0.619      94               8.49435
   2 │ 2      FIU_OLD      0.586      93               8.53072
   3 │ 3      FIU_OLD      0.584      95               8.54966
   4 │ 4      FIU_OLD      0.606      105              8.49505
   5 │ 5      FIU_OLD      0.619      79               8.50404
   6 │ 6      FIU_OLD      0.591      86               8.47885
   7 │ 7      FIU_OLD      0.611      79               8.4419
   8 │ 8      FIU_OLD      0.627      78               8.46411
   9 │ 9      FIU_OLD      0.608      91               8.4852
  10 │ 10     FIU_OLD      0.613      80               8.4755
  11 │ 11     FIU_OLD      0.606      79               8.50421
  12 │ 12     FIU_OLD      0.589      107              8.47449
  13 │ 13     FIU_

# FAU

In [16]:
desktop_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/FAU/"
degree_plan_file = desktop_path * "fau_degree_plan.csv"
grades_file      = desktop_path * "slim_grades_fau.csv"
vis_famu = desktop_path * "degree_plan_famu_vis.csv"
# -----------------------------------------------------------
slim_grade = CSV.read(grades_file, DataFrame) 
fau_plan = read_csv(degree_plan_file)
#vis_famu = read_csv(vis_famu)
#visualize(vis_famu)

DegreePlan("Generic", Curriculum(1528343564, "Computer Science - BS", "Florida Atlantic University", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(1528343564 => 1), "College Writing I", 3, String3("ENC"), String7("1101"), String3("FAU"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(2, Dict(1528343564 => 2), "Foundations of Computing", 3, String3("COP"), String7("1000"), String3("FAU"), "", "", Course[], String15("COT2000C"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(3, Dict(1528343564 => 3), "Introduction to Programming in Python", 3, String3("COP"), String7("2047"), String3("FAU"), "", "", Course[], String15("COP3035C"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(4, Dict(1528343564 => 4), "Calculus with Analytic Geometry 1", 4, String3("MAC"), String7("2311"), String3(

In [17]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES 
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

course_passrate = 0.9 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in fau_plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)
    
    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, slim_grade)

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

println("Pass rates successfully injected into the degree plan!")

--- INJECTING HISTORICAL PASS RATES ---
Pass rates successfully injected into the degree plan!


In [18]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")

# two ways for simulation
num_students = 1000
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis
# enroll sequentially through the courses in the degree plan unless they 

max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

 

simulation = simulate(fau_plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

--- RUNNING SIMULATION ---


Simulation(DegreePlan("Generic", Curriculum(1528343564, "Computer Science - BS", "Florida Atlantic University", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(1528343564 => 1), "College Writing I", 3, String3("ENC"), String7("1101"), String3("FAU"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}("model" => Dict{Any, Any}(:passrate => 0.8754689358587245), "termenrollment" => [1000.0, 103.0, 11.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "id" => 1, "failures" => 126, "num_students_passes" => 9568, "term_req" => 0, "grades" => [4.0, 0.0, 4.0, 4.0, 4.0, 4.0, 0.0, 4.0, 4.0, 4.0  …  4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 0.0, 4.0, 4.0], "enrolled" => 1114, "num_students_taken" => 10929, "students" => Student[]…), 0.8754689358587245), Course(2, Dict(1528343564 => 2), "Foundations of Computing", 3, String3("COP"), String7("1000"), String3("FAU"), "", "", Course[], String15("COT2000C"), Dict{Int64, Requisite}

In [ ]:
real_passrate = true   

simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)

### TEST - SIMULATION

In [19]:
for i in 1:100
    sim = simulate(fau_plan, max_course_attempts, cohort, max_credits = max_credits,  performance_model = PassRate, enrollment_model = Enrollment,
        duration = num_terms, duration_lock = duration_lock, stopouts = stopouts)
    push!(sim_df, [i, "FAU", sim.grad_rate, length(sim.reach_attempts_students), sim.time_to_degree])
end
# print(sim_df)

# FSU

In [20]:
desktop_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/FSU/"
degree_plan_file = desktop_path * "fsu_degree_plan.csv"
grades_file      = desktop_path * "slim_grades_fsu.csv"
vis_famu = desktop_path * "degree_plan_famu_vis.csv"
# -----------------------------------------------------------
slim_grade = CSV.read(grades_file, DataFrame) 
fsu_plan = read_csv(degree_plan_file)
#vis_famu = read_csv(vis_famu)
#visualize(vis_famu)

DegreePlan("Generic", Curriculum(731379530, "Computer Science - BS", "Florida State University", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(731379530 => 1), "Freshman Composition", 3, String3("ENC"), String7("1101"), String3("FSU"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(2, Dict(731379530 => 2), "Calculus with Analytic Geometry I", 4, String3("MAC"), String7("2311"), String3("FSU"), "", "", Course[], String15("MAC2311"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(3, Dict(731379530 => 3), "Introduction to C++", 3, String3("EEL"), String7("2880"), String3("FSU"), "", "", Course[], String15("COP3363"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(4, Dict(731379530 => 4), "General Education Core Social Science", 3, String3("SOC"), String7("UCC1"), String3("FSU"), "", 

In [21]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES 
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

course_passrate = 0.9 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in fsu_plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)
    
    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, slim_grade)

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

println("Pass rates successfully injected into the degree plan!")

--- INJECTING HISTORICAL PASS RATES ---
Pass rates successfully injected into the degree plan!


In [22]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")

# two ways for simulation
num_students = 1000
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis
# enroll sequentially through the courses in the degree plan unless they 

max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

 

simulation = simulate(fsu_plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

--- RUNNING SIMULATION ---


Simulation(DegreePlan("Generic", Curriculum(731379530, "Computer Science - BS", "Florida State University", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(731379530 => 1), "Freshman Composition", 3, String3("ENC"), String7("1101"), String3("FSU"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}("model" => Dict{Any, Any}(:passrate => 0.8754689358587245), "termenrollment" => [1000.0, 124.0, 15.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "id" => 1, "failures" => 152, "num_students_passes" => 9568, "term_req" => 0, "grades" => [4.0, 4.0, 4.0, 4.0, 4.0, 0.0, 4.0, 4.0, 4.0, 4.0  …  4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0], "enrolled" => 1139, "num_students_taken" => 10929, "students" => Student[]…), 0.8754689358587245), Course(2, Dict(731379530 => 2), "Calculus with Analytic Geometry I", 4, String3("MAC"), String7("2311"), String3("FSU"), "", "", Course[], String15("MAC2311"), Dict{Int64, Requi

In [ ]:
real_passrate = true   

simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)

### TEST - SIMULATION

In [23]:
for i in 1:100
    sim = simulate(plan, max_course_attempts, cohort, max_credits = max_credits,  performance_model = PassRate, enrollment_model = Enrollment,
        duration = num_terms, duration_lock = duration_lock, stopouts = stopouts)
    push!(sim_df, [i, "FSU", sim.grad_rate, length(sim.reach_attempts_students), sim.time_to_degree])
end
#print(sim_df)

# UCF

In [59]:
desktop_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/UCF/"
degree_plan_file = desktop_path * "ucf_degree_plan.csv"
grades_file      = desktop_path * "slim_grades_ucf.csv"
# -----------------------------------------------------------
slim_grade = CSV.read(grades_file, DataFrame) 
ucf_plan = read_csv(degree_plan_file)

DegreePlan("Generic", Curriculum(1087121741, "Computer Science - BS", "University of Central Florida", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(1087121741 => 1), "Composition I", 3, String3("ENC"), String7("1101"), String3("UCF"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(2, Dict(1087121741 => 2), "GEP Communications", 3, String3("SPC"), String7("UCC1"), String3("UCF"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(3, Dict(1087121741 => 3), "Introduction to Programming with C", 3, String3("EEL"), String7("2880"), String3("UCF"), "", "", Course[], String15("COP3223C"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(4, Dict(1087121741 => 4), "Calculus with Analytic Geometry I", 4, String3("MAC"), String7("2311"), String3("UCF"), "", "", Course[], Str

In [60]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES 
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

course_passrate = 0.9 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in ucf_plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)
    
    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, slim_grade)

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

println("Pass rates successfully injected into the degree plan!")

--- INJECTING HISTORICAL PASS RATES ---
Pass rates successfully injected into the degree plan!


In [61]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")

# two ways for simulation
num_students = 1000
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis
# enroll sequentially through the courses in the degree plan unless they 

max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

 

simulation = simulate(ucf_plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

--- RUNNING SIMULATION ---


Simulation(DegreePlan("Generic", Curriculum(1087121741, "Computer Science - BS", "University of Central Florida", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(1087121741 => 1), "Composition I", 3, String3("ENC"), String7("1101"), String3("UCF"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}("model" => Dict{Any, Any}(:passrate => 0.8754689358587245), "termenrollment" => [1000.0, 94.0, 9.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "id" => 1, "failures" => 117, "num_students_passes" => 9568, "term_req" => 0, "grades" => [4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0  …  0.0, 4.0, 0.0, 4.0, 0.0, 0.0, 4.0, 4.0, 4.0, 4.0], "enrolled" => 1103, "num_students_taken" => 10929, "students" => Student[]…), 0.8754689358587245), Course(2, Dict(1087121741 => 2), "GEP Communications", 3, String3("SPC"), String7("UCC1"), String3("UCF"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{

In [62]:
real_passrate = true   

simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)


------------ Simulation Report ------------
Computer Science - BS, BS -- Generic

-------- Simulation Statistics --------
Number of terms: 12
Max Credits per Term: 18
Max Course Attempts: 3
Number of Students: 1000
Preset Course Pass Rates: 90.0%

-------- Graduation Statistics --------
Number of Students Graduated: 494
Graduation Rate: 49.4%
Term Graduation Rates: 
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.154, 0.418, 0.49, 0.494, 0.0]
Average time to degree: 8.850202429149798 terms

-------- Stop out Statistics --------
Number of Students Stopped Out (Stopout Model Prediction + Reached Max Attempts): 506
Number of Students Reaching Max Attempts: 200
Stop-out Rate: 50.6%
Cumulative Term Stop-out Rates (including reached max course attempts students): 
[0.079, 0.204, 0.278, 0.347, 0.423, 0.464, 0.479, 0.484, 0.494, 0.505, 0.506, 0.0]

Cumulative Term Stop-out Rates (excluding reaching max course attempts students): 
[0.079, 0.204, 0.24, 0.275, 0.297, 0.3, 0.306, 0.306, 0.306, 0.306, 0.306

## TEST - SIMULATION

In [27]:
for i in 1:100
    sim = simulate(ucf_plan, max_course_attempts, cohort, max_credits = max_credits,  performance_model = PassRate, enrollment_model = Enrollment,
        duration = num_terms, duration_lock = duration_lock, stopouts = stopouts)
    push!(sim_df, [i, "UCF", sim.grad_rate, length(sim.reach_attempts_students), sim.time_to_degree])
end
# print(sim_df)

600×5 DataFrame
 Row │ SimID  Institution  Grad_Rate  Std_Max_Attempt  Time_To_Degree 
     │ Any    Any          Any        Any              Any            
─────┼────────────────────────────────────────────────────────────────
   1 │ 1      FIU_OLD      0.619      94               8.49435
   2 │ 2      FIU_OLD      0.586      93               8.53072
   3 │ 3      FIU_OLD      0.584      95               8.54966
   4 │ 4      FIU_OLD      0.606      105              8.49505
   5 │ 5      FIU_OLD      0.619      79               8.50404
   6 │ 6      FIU_OLD      0.591      86               8.47885
   7 │ 7      FIU_OLD      0.611      79               8.4419
   8 │ 8      FIU_OLD      0.627      78               8.46411
   9 │ 9      FIU_OLD      0.608      91               8.4852
  10 │ 10     FIU_OLD      0.613      80               8.4755
  11 │ 11     FIU_OLD      0.606      79               8.50421
  12 │ 12     FIU_OLD      0.589      107              8.47449
  13 │ 13     FIU_

# UF

In [28]:
desktop_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/UF/"
degree_plan_file = desktop_path * "uf_degree_plan.csv"
grades_file      = desktop_path * "slim_grades_uf.csv"
# -----------------------------------------------------------
slim_grade = CSV.read(grades_file, DataFrame) 
uf_plan = read_csv(degree_plan_file)

DegreePlan("Generic", Curriculum(1973106718, "Computer Science - BS", "University of Florida", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(1973106718 => 1), "General Education: Humanities", 3, String3("HUM"), String7("UCC1"), String3("UF"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(2, Dict(1973106718 => 2), "Introduction to Principles of Physics", 3, String3("PHY"), String7("1020"), String3("UF"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(3, Dict(1973106718 => 3), "Programming Fundamentals 1", 4, String3("COP"), String7("2210"), String3("UF"), "", "", Course[], String15("COP3502C"), Dict{Int64, Requisite}(4 => co), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(4, Dict(1973106718 => 4), "Analytic Geometry and Calculus 1", 4, String3("MAC"), String7("2311"), String3("UF"), 

In [29]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES 
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

course_passrate = 0.9 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in uf_plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)
    
    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, slim_grade)

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

println("Pass rates successfully injected into the degree plan!")

--- INJECTING HISTORICAL PASS RATES ---
Pass rates successfully injected into the degree plan!


In [30]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")

# two ways for simulation
num_students = 1000
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis
# enroll sequentially through the courses in the degree plan unless they 

max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

 

simulation = simulate(uf_plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

--- RUNNING SIMULATION ---


Simulation(DegreePlan("Generic", Curriculum(1973106718, "Computer Science - BS", "University of Florida", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(1973106718 => 1), "General Education: Humanities", 3, String3("HUM"), String7("UCC1"), String3("UF"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}("model" => Dict{Any, Any}(:passrate => 0.8987461308430827), "termenrollment" => [1000.0, 91.0, 12.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "id" => 1, "failures" => 111, "num_students_passes" => 17131, "term_req" => 0, "grades" => [4.0, 4.0, 4.0, 0.0, 0.0, 4.0, 4.0, 4.0, 4.0, 4.0  …  4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0], "enrolled" => 1103, "num_students_taken" => 19061, "students" => Student[]…), 0.8987461308430827), Course(2, Dict(1973106718 => 2), "Introduction to Principles of Physics", 3, String3("PHY"), String7("1020"), String3("UF"), "", "", Course[], "", Dict{Int64, Requisite}(

In [ ]:
real_passrate = true   

simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)

### TEST - SIMULATION

In [31]:
for i in 1:100
    sim = simulate(uf_plan, max_course_attempts, cohort, max_credits = max_credits,  performance_model = PassRate, enrollment_model = Enrollment,
        duration = num_terms, duration_lock = duration_lock, stopouts = stopouts)
    push!(sim_df, [i, "UF", sim.grad_rate, length(sim.reach_attempts_students), sim.time_to_degree])
end
print(sim_df)

700×5 DataFrame
 Row │ SimID  Institution  Grad_Rate  Std_Max_Attempt  Time_To_Degree 
     │ Any    Any          Any        Any              Any            
─────┼────────────────────────────────────────────────────────────────
   1 │ 1      FIU_OLD      0.619      94               8.49435
   2 │ 2      FIU_OLD      0.586      93               8.53072
   3 │ 3      FIU_OLD      0.584      95               8.54966
   4 │ 4      FIU_OLD      0.606      105              8.49505
   5 │ 5      FIU_OLD      0.619      79               8.50404
   6 │ 6      FIU_OLD      0.591      86               8.47885
   7 │ 7      FIU_OLD      0.611      79               8.4419
   8 │ 8      FIU_OLD      0.627      78               8.46411
   9 │ 9      FIU_OLD      0.608      91               8.4852
  10 │ 10     FIU_OLD      0.613      80               8.4755
  11 │ 11     FIU_OLD      0.606      79               8.50421
  12 │ 12     FIU_OLD      0.589      107              8.47449
  13 │ 13     FIU_

# UNF

In [32]:
desktop_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/UNF/"
degree_plan_file = desktop_path * "unf_degree_plan.csv"
grades_file      = desktop_path * "slim_grades_unf.csv"
# -----------------------------------------------------------
slim_grade = CSV.read(grades_file, DataFrame) 
unf_plan = read_csv(degree_plan_file)

DegreePlan("Generic", Curriculum(192483264, "Computer Science - BS", "University of North Florida", "BS", semester, "11.0701", AbstractCourse[Course(1, Dict(192483264 => 1), "Audience and Purpose", 3, String3("ENC"), String7("1101"), String3("UNF"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(2, Dict(192483264 => 2), "Calculus I", 4, String3("MAC"), String7("2311"), String3("UNF"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(3, Dict(192483264 => 3), "Science Course for Science Majors", 3, String3("NAT"), String7("UCC1"), String3("UNF"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(4, Dict(192483264 => 4), "Beauty and Joy of Computing", 3, String3("IDC"), String7("1000"), String3("UNF"), "", "", Course[], String15("IDC2000"), Dict{Int64, Req

In [33]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES 
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

course_passrate = 0.9 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in unf_plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)
    
    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, slim_grade)

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

println("Pass rates successfully injected into the degree plan!")

--- INJECTING HISTORICAL PASS RATES ---
Pass rates successfully injected into the degree plan!


In [34]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")

# two ways for simulation
num_students = 1000
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis
# enroll sequentially through the courses in the degree plan unless they 

max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

 

simulation = simulate(unf_plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

--- RUNNING SIMULATION ---


Simulation(DegreePlan("Generic", Curriculum(192483264, "Computer Science - BS", "University of North Florida", "BS", semester, "11.0701", AbstractCourse[Course(1, Dict(192483264 => 1), "Audience and Purpose", 3, String3("ENC"), String7("1101"), String3("UNF"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}("model" => Dict{Any, Any}(:passrate => 0.8754689358587245), "termenrollment" => [1000.0, 109.0, 15.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "id" => 1, "failures" => 139, "num_students_passes" => 9568, "term_req" => 0, "grades" => [4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0  …  4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0], "enrolled" => 1124, "num_students_taken" => 10929, "students" => Student[]…), 0.8754689358587245), Course(2, Dict(192483264 => 2), "Calculus I", 4, String3("MAC"), String7("2311"), String3("UNF"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{Stri

In [ ]:
real_passrate = true   

simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)

### TEST - SIMULATION

In [35]:
for i in 1:100
    sim = simulate(unf_plan, max_course_attempts, cohort, max_credits = max_credits,  performance_model = PassRate, enrollment_model = Enrollment,
        duration = num_terms, duration_lock = duration_lock, stopouts = stopouts)
    push!(sim_df, [i, "UNF", sim.grad_rate, length(sim.reach_attempts_students), sim.time_to_degree])
end
print(sim_df)

800×5 DataFrame
 Row │ SimID  Institution  Grad_Rate  Std_Max_Attempt  Time_To_Degree 
     │ Any    Any          Any        Any              Any            
─────┼────────────────────────────────────────────────────────────────
   1 │ 1      FIU_OLD      0.619      94               8.49435
   2 │ 2      FIU_OLD      0.586      93               8.53072
   3 │ 3      FIU_OLD      0.584      95               8.54966
   4 │ 4      FIU_OLD      0.606      105              8.49505
   5 │ 5      FIU_OLD      0.619      79               8.50404
   6 │ 6      FIU_OLD      0.591      86               8.47885
   7 │ 7      FIU_OLD      0.611      79               8.4419
   8 │ 8      FIU_OLD      0.627      78               8.46411
   9 │ 9      FIU_OLD      0.608      91               8.4852
  10 │ 10     FIU_OLD      0.613      80               8.4755
  11 │ 11     FIU_OLD      0.606      79               8.50421
  12 │ 12     FIU_OLD      0.589      107              8.47449
  13 │ 13     FIU_

# USF

In [36]:
desktop_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/USF/"
degree_plan_file = desktop_path * "usf_degree_plan.csv"
grades_file      = desktop_path * "slim_grades_usf.csv"
# -----------------------------------------------------------
slim_grade = CSV.read(grades_file, DataFrame) 
usf_plan = read_csv(degree_plan_file)

DegreePlan("Generic", Curriculum(1399185057, "Computer Science - BS", "University of South Florida", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(1399185057 => 1), "Freshman Seminar for Computing", 0, String3("CGS"), String7("1920"), String3("USF"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(2, Dict(1399185057 => 2), "Programming Concepts", 3, String3("COP"), String7("2210"), String3("USF"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(3, Dict(1399185057 => 3), "Composition I", 3, String3("ENC"), String7("1101"), String3("USF"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(4, Dict(1399185057 => 4), "Calculus I", 4, String3("MAC"), String7("2311"), String3("USF"), "", "", Course[], String15("IDC2000"), Dict{Int64, Requisite}(), L

In [37]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES 
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

course_passrate = 0.9 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in usf_plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)
    
    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, slim_grade)

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

println("Pass rates successfully injected into the degree plan!")

--- INJECTING HISTORICAL PASS RATES ---
Pass rates successfully injected into the degree plan!


In [38]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")

# two ways for simulation
num_students = 1000
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis
# enroll sequentially through the courses in the degree plan unless they 

max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

 

simulation = simulate(usf_plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

--- RUNNING SIMULATION ---


Simulation(DegreePlan("Generic", Curriculum(1399185057, "Computer Science - BS", "University of South Florida", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(1399185057 => 1), "Freshman Seminar for Computing", 0, String3("CGS"), String7("1920"), String3("USF"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}("model" => Dict{Any, Any}(:passrate => 0.9459459459459459), "termenrollment" => [1000.0, 61.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "id" => 1, "failures" => 67, "num_students_passes" => 1365, "term_req" => 0, "grades" => [4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0  …  4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0], "enrolled" => 1062, "num_students_taken" => 1443, "students" => Student[]…), 0.9459459459459459), Course(2, Dict(1399185057 => 2), "Programming Concepts", 3, String3("COP"), String7("2210"), String3("USF"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningO

In [ ]:
real_passrate = true   

simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)

In [39]:
for i in 1:100
    sim = simulate(usf_plan, max_course_attempts, cohort, max_credits = max_credits,  performance_model = PassRate, enrollment_model = Enrollment,
        duration = num_terms, duration_lock = duration_lock, stopouts = stopouts)
    push!(sim_df, [i, "USF", sim.grad_rate, length(sim.reach_attempts_students), sim.time_to_degree])
end
print(sim_df)

900×5 DataFrame
 Row │ SimID  Institution  Grad_Rate  Std_Max_Attempt  Time_To_Degree 
     │ Any    Any          Any        Any              Any            
─────┼────────────────────────────────────────────────────────────────
   1 │ 1      FIU_OLD      0.619      94               8.49435
   2 │ 2      FIU_OLD      0.586      93               8.53072
   3 │ 3      FIU_OLD      0.584      95               8.54966
   4 │ 4      FIU_OLD      0.606      105              8.49505
   5 │ 5      FIU_OLD      0.619      79               8.50404
   6 │ 6      FIU_OLD      0.591      86               8.47885
   7 │ 7      FIU_OLD      0.611      79               8.4419
   8 │ 8      FIU_OLD      0.627      78               8.46411
   9 │ 9      FIU_OLD      0.608      91               8.4852
  10 │ 10     FIU_OLD      0.613      80               8.4755
  11 │ 11     FIU_OLD      0.606      79               8.50421
  12 │ 12     FIU_OLD      0.589      107              8.47449
  13 │ 13     FIU_

# UWF

In [40]:
desktop_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/UWF/"
degree_plan_file = desktop_path * "uwf_degree_plan.csv"
grades_file      = desktop_path * "slim_grades_uwf.csv"
# -----------------------------------------------------------
slim_grade = CSV.read(grades_file, DataFrame) 
uwf_plan = read_csv(degree_plan_file)

DegreePlan("Generic", Curriculum(3040264664, "Computer Science - BS", "University of West Florida", "BS", semester, "11.0701", AbstractCourse[Course(1, Dict(3040264664 => 1), "English Composition I", 3, String3("ENC"), String7("1101"), String3("UWF"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(2, Dict(3040264664 => 2), "General Education: Communication", 4, String3("SPC"), String7("2608"), String3("UWF"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(3, Dict(3040264664 => 3), "Analytic Geometry and Calculus I", 3, String3("MAC"), String7("2311"), String3("UWF"), "", "", Course[], "", Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(4, Dict(3040264664 => 4), "Introductory Programming ", 3, String3("COP"), String7("2210"), String3("UWF"), "", "", Course[], String15("ID

In [41]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES 
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

course_passrate = 0.9 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in uwf_plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)
    
    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, slim_grade)

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

println("Pass rates successfully injected into the degree plan!")

--- INJECTING HISTORICAL PASS RATES ---
Pass rates successfully injected into the degree plan!


In [42]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")

# two ways for simulation
num_students = 1000
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis
# enroll sequentially through the courses in the degree plan unless they 

max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

 

simulation = simulate(uwf_plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

--- RUNNING SIMULATION ---


Simulation(DegreePlan("Generic", Curriculum(3040264664, "Computer Science - BS", "University of West Florida", "BS", semester, "11.0701", AbstractCourse[Course(1, Dict(3040264664 => 1), "English Composition I", 3, String3("ENC"), String7("1101"), String3("UWF"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}("model" => Dict{Any, Any}(:passrate => 0.8754689358587245), "termenrollment" => [1000.0, 95.0, 8.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "id" => 1, "failures" => 122, "num_students_passes" => 9568, "term_req" => 0, "grades" => [4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0  …  4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0], "enrolled" => 1103, "num_students_taken" => 10929, "students" => Student[]…), 0.8754689358587245), Course(2, Dict(3040264664 => 2), "General Education: Communication", 4, String3("SPC"), String7("2608"), String3("UWF"), "", "", Course[], "", Dict{Int64, Requisite}(), Learn

In [ ]:
real_passrate = true   

simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)

### TEST - SIMULATION

In [43]:
for i in 1:100
    sim = simulate(uwf_plan, max_course_attempts, cohort, max_credits = max_credits,  performance_model = PassRate, enrollment_model = Enrollment,
        duration = num_terms, duration_lock = duration_lock, stopouts = stopouts)
    push!(sim_df, [i, "UWF", sim.grad_rate, length(sim.reach_attempts_students), sim.time_to_degree])
end
print(sim_df)

1000×5 DataFrame
  Row │ SimID  Institution  Grad_Rate  Std_Max_Attempt  Time_To_Degree 
      │ Any    Any          Any        Any              Any            
──────┼────────────────────────────────────────────────────────────────
    1 │ 1      FIU_OLD      0.619      94               8.49435
    2 │ 2      FIU_OLD      0.586      93               8.53072
    3 │ 3      FIU_OLD      0.584      95               8.54966
    4 │ 4      FIU_OLD      0.606      105              8.49505
    5 │ 5      FIU_OLD      0.619      79               8.50404
    6 │ 6      FIU_OLD      0.591      86               8.47885
    7 │ 7      FIU_OLD      0.611      79               8.4419
    8 │ 8      FIU_OLD      0.627      78               8.46411
    9 │ 9      FIU_OLD      0.608      91               8.4852
   10 │ 10     FIU_OLD      0.613      80               8.4755
   11 │ 11     FIU_OLD      0.606      79               8.50421
   12 │ 12     FIU_OLD      0.589      107              8.47449
  

# FGCU

In [44]:
desktop_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/FGCU/"
degree_plan_file = desktop_path * "fgcu_degree_plan.csv"
grades_file      = desktop_path * "slim_grades_fgcu.csv"
# -----------------------------------------------------------
slim_grade = CSV.read(grades_file, DataFrame) 
fgcu_plan = read_csv(degree_plan_file)

DegreePlan("Generic", Curriculum(4247769023, "Computer Science - BS", "Florida Gulf Coast University", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(4247769023 => 1), "Writing and Rhetoric I", 3, String15("ENC"), String7("1101"), String7("FGCU"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(2, Dict(4247769023 => 2), "Calculus I", 4, String15("MAC"), String7("2311"), String7("FGCU"), "", "", Course[], String15("MAC2311"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(3, Dict(4247769023 => 3), "Introduction to Computer Science", 3, String15("COP"), String7("1000"), String7("FGCU"), "", "", Course[], String15("COP1500"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(4, Dict(4247769023 => 4), "General Elective: Social Science", 3, String15("SOC"), String7("UCC2"), String7("FGCU"),

In [45]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES 
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

course_passrate = 0.9 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in fgcu_plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)
    
    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, slim_grade)

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

println("Pass rates successfully injected into the degree plan!")

--- INJECTING HISTORICAL PASS RATES ---
Pass rates successfully injected into the degree plan!


In [46]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")

# two ways for simulation
num_students = 1000
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis
# enroll sequentially through the courses in the degree plan unless they 

max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

 

simulation = simulate(fgcu_plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

--- RUNNING SIMULATION ---


Simulation(DegreePlan("Generic", Curriculum(4247769023, "Computer Science - BS", "Florida Gulf Coast University", "BS", semester, "11.0101", AbstractCourse[Course(1, Dict(4247769023 => 1), "Writing and Rhetoric I", 3, String15("ENC"), String7("1101"), String7("FGCU"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}("model" => Dict{Any, Any}(:passrate => 0.8754689358587245), "termenrollment" => [1000.0, 108.0, 14.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "id" => 1, "failures" => 132, "num_students_passes" => 9568, "term_req" => 0, "grades" => [0.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 0.0  …  4.0, 4.0, 0.0, 4.0, 4.0, 0.0, 0.0, 4.0, 4.0, 4.0], "enrolled" => 1122, "num_students_taken" => 10929, "students" => Student[]…), 0.8754689358587245), Course(2, Dict(4247769023 => 2), "Calculus I", 4, String15("MAC"), String7("2311"), String7("FGCU"), "", "", Course[], String15("MAC2311"), Dict{Int64, Requisite}(), 

In [ ]:
real_passrate = true   

simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)

### TEST - SIMULATION

In [47]:
for i in 1:100
    sim = simulate(fgcu_plan, max_course_attempts, cohort, max_credits = max_credits,  performance_model = PassRate, enrollment_model = Enrollment,
        duration = num_terms, duration_lock = duration_lock, stopouts = stopouts)
    push!(sim_df, [i, "FGCU", sim.grad_rate, length(sim.reach_attempts_students), sim.time_to_degree])
end
#print(sim_df)

1100×5 DataFrame
  Row │ SimID  Institution  Grad_Rate  Std_Max_Attempt  Time_To_Degree 
      │ Any    Any          Any        Any              Any            
──────┼────────────────────────────────────────────────────────────────
    1 │ 1      FIU_OLD      0.619      94               8.49435
    2 │ 2      FIU_OLD      0.586      93               8.53072
    3 │ 3      FIU_OLD      0.584      95               8.54966
    4 │ 4      FIU_OLD      0.606      105              8.49505
    5 │ 5      FIU_OLD      0.619      79               8.50404
    6 │ 6      FIU_OLD      0.591      86               8.47885
    7 │ 7      FIU_OLD      0.611      79               8.4419
    8 │ 8      FIU_OLD      0.627      78               8.46411
    9 │ 9      FIU_OLD      0.608      91               8.4852
   10 │ 10     FIU_OLD      0.613      80               8.4755
   11 │ 11     FIU_OLD      0.606      79               8.50421
   12 │ 12     FIU_OLD      0.589      107              8.47449
  

# FPU

In [48]:
desktop_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/FPU/"
degree_plan_file = desktop_path * "fpu_degree_plan.csv"
grades_file      = desktop_path * "slim_grades_fpu.csv"
# -----------------------------------------------------------
slim_grade = CSV.read(grades_file, DataFrame) 
fpu_plan = read_csv(degree_plan_file)

DegreePlan("Generic", Curriculum(937411817, "Computer Science - BS", "Florida Polytechnic University", "BS", semester, "11.0701", AbstractCourse[Course(1, Dict(937411817 => 1), "Beauty and Joy of Computing", 2, String3("COP"), String7("1000"), String3("FPU"), "", "", Course[], String15("COP1009"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(2, Dict(937411817 => 2), "Analytic Geometry and Calculus 1", 4, String3("MAC"), String7("2311"), String3("FPU"), "", "", Course[], String15("MAC2311"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(3, Dict(937411817 => 3), "Expository and Argumentative Writing", 3, String3("ENC"), String7("1101"), String3("FPU"), "", "", Course[], String15("ENC1101"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}(), 0.5), Course(4, Dict(937411817 => 4), "General Education: Humanities", 3, String3("HUM"), String7("UCC1"),

In [49]:
# ---------------------------------------------------------
# 2. INJECT HISTORICAL PASS RATES 
# ---------------------------------------------------------
println("--- INJECTING HISTORICAL PASS RATES ---")

course_passrate = 0.9 # Fallback
passing_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "P"]
all_grades = ["C+", "A", "F", "B-", "P", "B", "B+", "A-", "C", "DR", "IN", "D", "F0", "AU", "NC", "WI", "NG", "+", "C-", "NR", "H"]

clean_course(x) = replace(uppercase(strip(string(ismissing(x) ? "" : x))), r"[^A-Z0-9]" => "")
clean_grade(x)  = strip(string(ismissing(x) ? "" : x))

# CRITICAL FIX: Loop over plan.curriculum.courses 
for course in fpu_plan.curriculum.courses
    p = clean_course(course.prefix)
    n = clean_course(course.num)
    
    course_data = filter(row -> clean_course(row.course_prefix) == p && clean_course(row.course_num) == n, slim_grade)

    num_students_taken = nrow(filter(row -> clean_grade(row.grade) in all_grades, course_data))
    num_passes = nrow(filter(row -> clean_grade(row.grade) in passing_grades, course_data))

    if num_students_taken == 0
        course.passrate = course_passrate 
        course.metadata["num_students_taken"] = "N/A"
        course.metadata["num_students_passes"] = "N/A"
    else
        course.passrate = num_passes / num_students_taken
        course.metadata["num_students_taken"] = num_students_taken
        course.metadata["num_students_passes"] = num_passes
    end
end

println("Pass rates successfully injected into the degree plan!")

--- INJECTING HISTORICAL PASS RATES ---
Pass rates successfully injected into the degree plan!


In [50]:
# ---------------------------------------------------------
# 3. RUN THE SIMULATION
# ---------------------------------------------------------
println("--- RUNNING SIMULATION ---")

# two ways for simulation
num_students = 1000
cohort = simple_students(num_students) # starts 1000 students at term 1, rolls 0 to 100, if it is >= to pass rate, they pass on a percentage basis
# enroll sequentially through the courses in the degree plan unless they 

max_course_attempts = 3  
num_terms = 12           
max_credits = 18
duration_lock = false
stopouts = true # graduation attrition 

 

simulation = simulate(fpu_plan, max_course_attempts, cohort,
                      max_credits = max_credits,
                      performance_model = PassRate,
                      enrollment_model = Enrollment,
                      duration = num_terms,
                      duration_lock = duration_lock,
                      stopouts = stopouts)

--- RUNNING SIMULATION ---


Simulation(DegreePlan("Generic", Curriculum(937411817, "Computer Science - BS", "Florida Polytechnic University", "BS", semester, "11.0701", AbstractCourse[Course(1, Dict(937411817 => 1), "Beauty and Joy of Computing", 2, String3("COP"), String7("1000"), String3("FPU"), "", "", Course[], String15("COP1009"), Dict{Int64, Requisite}(), LearningOutcome[], Dict{String, Any}(), Dict{String, Any}("model" => Dict{Any, Any}(:passrate => 0.8042553191489362), "termenrollment" => [1000.0, 189.0, 30.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "id" => 1, "failures" => 247, "num_students_passes" => 189, "term_req" => 0, "grades" => [4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 0.0, 4.0, 0.0  …  0.0, 4.0, 0.0, 4.0, 0.0, 0.0, 4.0, 4.0, 4.0, 4.0], "enrolled" => 1219, "num_students_taken" => 235, "students" => Student[]…), 0.8042553191489362), Course(2, Dict(937411817 => 2), "Analytic Geometry and Calculus 1", 4, String3("MAC"), String7("2311"), String3("FPU"), "", "", Course[], String15("MAC2311"), Dict{Int

In [ ]:
real_passrate = true   

simulation_report(simulation, num_terms, course_passrate, max_credits, real_passrate)

### TEST - SIMULATION

In [51]:
for i in 1:100
    sim = simulate(fpu_plan, max_course_attempts, cohort, max_credits = max_credits,  performance_model = PassRate, enrollment_model = Enrollment,
        duration = num_terms, duration_lock = duration_lock, stopouts = stopouts)
    push!(sim_df, [i, "FPU", sim.grad_rate, length(sim.reach_attempts_students), sim.time_to_degree])
end
print(sim_df)

1200×5 DataFrame
  Row │ SimID  Institution  Grad_Rate  Std_Max_Attempt  Time_To_Degree 
      │ Any    Any          Any        Any              Any            
──────┼────────────────────────────────────────────────────────────────
    1 │ 1      FIU_OLD      0.619      94               8.49435
    2 │ 2      FIU_OLD      0.586      93               8.53072
    3 │ 3      FIU_OLD      0.584      95               8.54966
    4 │ 4      FIU_OLD      0.606      105              8.49505
    5 │ 5      FIU_OLD      0.619      79               8.50404
    6 │ 6      FIU_OLD      0.591      86               8.47885
    7 │ 7      FIU_OLD      0.611      79               8.4419
    8 │ 8      FIU_OLD      0.627      78               8.46411
    9 │ 9      FIU_OLD      0.608      91               8.4852
   10 │ 10     FIU_OLD      0.613      80               8.4755
   11 │ 11     FIU_OLD      0.606      79               8.50421
   12 │ 12     FIU_OLD      0.589      107              8.47449
  

In [52]:
simulation_results_path = "C:/Users/cpanc/OneDrive/Documents/IDC6940/" # <- get sim at very end!
all_unis = simulation_results_path * "all_uni_sim_results.csv"
CSV.write(all_unis, sim_df) 

"C:/Users/cpanc/OneDrive/Documents/IDC6940/all_uni_sim_results.csv"